In [ ]:
import os
import numpy as np
import pandas as pd
from keras.preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.models import Model
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

train_dir = 'MRNet-v1.0/train'
valid_dir = 'MRNet-v1.0/valid'
label_csv_files = ['MRNet-v1.0/train-acl.csv', 'MRNet-v1.0/valid-acl.csv']

def load_npy_images(base_dir):
    images = []
    filenames = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.npy'):
                img = np.load(os.path.join(root, file))
                images.append(img)
                filenames.append(file)
    return images, filenames

def load_labels(label_csv_files):
    labels_dict = {}
    for label_file in label_csv_files:
        if os.path.exists(label_file):
            df = pd.read_csv(label_file, header=None, names=['index', 'label'])
            for idx, row in df.iterrows():
                filename = f"{str(int(row['index'])).zfill(4)}.npy"
                labels_dict[filename] = row['label']
        else:
            print(f"File {label_file} does not exist")
    return labels_dict

train_images, train_filenames = load_npy_images(train_dir)
valid_images, valid_filenames = load_npy_images(valid_dir)
labels_dict = load_labels(label_csv_files)

def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        img_array = img_array[0]
    return img_array

def extract_sift_features(img_array):
    sift = cv2.SIFT_create()
    keypoints, descriptors = sift.detectAndCompute(img_array, None)
    if descriptors is None:
        descriptors = np.zeros((1, 128))
    flat_features = descriptors.flatten()
    return flat_features

features_list = []
labels_list = []
filenames_list = []

def process_images(images, filenames):
    for img, filename in zip(images, filenames):
        img = preprocess_image(img)
        features = extract_sift_features(img)
        features_list.append(features)
        label = labels_dict.get(filename, 'Unknown')
        labels_list.append(label)
        filenames_list.append(filename)
        print(f"Processed {filename}: label = {label}")

process_images(train_images, train_filenames)
process_images(valid_images, valid_filenames)

max_len = max([len(f) for f in features_list])
features_list = [np.pad(f, (0, max_len - len(f)), mode='constant') for f in features_list]

features_df = pd.DataFrame(features_list)
features_df.insert(0, 'filename', filenames_list)
features_df.insert(1, 'label', labels_list)

features_df.to_csv('extracted_feature_SIFT.csv', index=False)
print("Features and labels saved to extracted_feature_SIFT.csv")


In [ ]:
features_df = pd.read_csv('extracted_feature_SIFT.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print(f"SVM Accuracy: {accuracy_svm}")
print(f"SVM Precision: {precision_svm}")
print(f"SVM Recall: {recall_svm}")
print(f"SVM F1 Score: {f1_svm}")

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn))
print(f"KNN Accuracy: {accuracy_knn}")
print(f"KNN Precision: {precision_knn}")
print(f"KNN Recall: {recall_knn}")
print(f"KNN F1 Score: {f1_knn}")

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))
print(f"Decision Tree Accuracy: {accuracy_dt}")
print(f"Decision Tree Precision: {precision_dt}")
print(f"Decision Tree Recall: {recall_dt}")
print(f"Decision Tree F1 Score: {f1_dt}")

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

features_df = pd.read_csv('extracted_feature_SIFT.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print(f"SVM Accuracy: {accuracy_svm}")
print(f"SVM Precision: {precision_svm}")
print(f"SVM Recall: {recall_svm}")
print(f"SVM F1 Score: {f1_svm}")

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn))
print(f"KNN Accuracy: {accuracy_knn}")
print(f"KNN Precision: {precision_knn}")
print(f"KNN Recall: {recall_knn}")
print(f"KNN F1 Score: {f1_knn}")

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))
print(f"Decision Tree Accuracy: {accuracy_dt}")
print(f"Decision Tree Precision: {precision_dt}")
print(f"Decision Tree Recall: {recall_dt}")
print(f"Decision Tree F1 Score: {f1_dt}")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from tqdm import tqdm

print("Loading data...")
features_df = pd.read_csv('extracted_feature_SIFT.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X,  y, test_size=0.2, random_state=42)

svm_model = SVC(probability=True)
knn_model = KNeighborsClassifier(n_neighbors=5)
dt_model = DecisionTreeClassifier()

ensemble_model = VotingClassifier(
    estimators=[('svm', svm_model), ('knn', knn_model), ('dt', dt_model)],
    voting='soft'
)

print("Training ensemble model...")
ensemble_model.fit(X_train, y_train)

print("Making predictions on training data...")
y_pred_train_ensemble = ensemble_model.predict(X_train)

print("Making predictions on test data...")
y_pred_test_ensemble = ensemble_model.predict(X_test)

train_accuracy_ensemble = accuracy_score(y_train, y_pred_train_ensemble)
test_accuracy_ensemble = accuracy_score(y_test, y_pred_test_ensemble)
test_recall_ensemble = recall_score(y_test, y_pred_test_ensemble, average='weighted')
test_f1_ensemble = f1_score(y_test, y_pred_test_ensemble, average='weighted')

print("\nEnsemble Model (VotingClassifier) Classification Report:")
print(classification_report(y_test, y_pred_test_ensemble))
print(f"Ensemble Training Accuracy: {train_accuracy_ensemble}")
print(f"Ensemble Test Accuracy: {test_accuracy_ensemble}")
print(f"Ensemble Test Recall: {test_recall_ensemble}")
print(f"Ensemble Test F1 Score: {test_f1_ensemble}")


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

npy_file_path = 'MRNet-v1.0/train/axial/0001.npy'
image_array = np.load(npy_file_path)

if len(image_array.shape) == 3 and image_array.shape[0] > 1:
    image_array = image_array[image_array.shape[0] // 2]

image_array_uint8 = image_array.astype(np.uint8)

def apply_sift(image):
    if len(image.shape) == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray_image = image
    
    sift = cv2.SIFT_create()
    
    keypoints, descriptors = sift.detectAndCompute(gray_image, None)
    
    sift_image = cv2.drawKeypoints(gray_image, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    
    return sift_image

sift_image = apply_sift(image_array_uint8)

def show_image(img_array, title, cmap='gray'):
    plt.imshow(img_array, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_image(image_array_uint8, "Original Image", cmap='gray')
show_image(sift_image, "Image After SIFT Processing", cmap='gray')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

npy_file_path_sagittal = 'MRNet-v1.0/train/sagittal/0001.npy'
npy_file_path_axial = 'MRNet-v1.0/train/axial/0001.npy'
npy_file_path_coronal = 'MRNet-v1.0/train/coronal/0001.npy'

images_sagittal = np.load(npy_file_path_sagittal)
images_axial = np.load(npy_file_path_axial)
images_coronal = np.load(npy_file_path_coronal)

def apply_sift(image):
    image_uint8 = image.astype(np.uint8)
    
    if len(image.shape) == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2GRAY)
    else:
        gray_image = image_uint8
    
    sift = cv2.SIFT_create()
    
    keypoints, descriptors = sift.detectAndCompute(gray_image, None)
    
    sift_image = cv2.drawKeypoints(gray_image, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    
    return sift_image

def display_images(images, title_prefix='Image', num_images=1):
    plt.figure(figsize=(15, 5))
    plt.subplot(1, num_images, i + 1)
    img = images
    plt.imshow(img, cmap='gray')
    plt.title(f'Original Data')
    plt.axis('off')
    plt.show()


display_images(images_sagittal, title_prefix='Sagittal')
display_images(images_axial, title_prefix='Axial')
display_images(images_coronal, title_prefix='Coronal')
